# Kagle inclass https://www.kaggle.com/c/simplesentiment/overview

In [1]:
from itertools import product
from collections import Counter
import os
import re
import pickle as pkl

import nltk
from nltk.corpus import stopwords
import nltk.stem as st
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression#, SGDClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from prepare_text_class import TextPrepareClass
from utilities import seedEverything

In [5]:
rand_seed = 141592
seedEverything(rand_seed)

## Задаю переменные

In [6]:
PATH_DATA = os.path.join(Path.cwd(), 'data')
PATH_SUBM = os.path.join(Path.cwd(), 'submissions')
PATH_MODL = os.path.join(Path.cwd(), 'models')

In [7]:
PHONES = r'\b(samsung|galaxy|xiaomi|iphon|redmi|note|honor|huawei|apple|' +\
          'nokia|meizu|google|самсунг|айфон|mi|lenovo|lg|redme|asus|vivo|' +\
          'zte|helio|mediatek|oppo|htc|pixel|xperia|fly|realme|zenfone|' +\
          'alcatel|blade|philips|touch|lumia|oneplus|motorola|inoi|red|neo|' +\
          'moto|panasonic|band|honnor|bbk|vertex|lafleur|xiomi|редми|хонор|' +\
          'ноки|хуаве|мейзу|асус|галакси|иной|гэлакси|хонор)(|[a-zа-я]+)\b'

## Загрузка, очистка и преобразование данных

In [8]:
df = pd.read_csv(os.path.join(PATH_DATA, 'ru_train.csv'), )
df.shape

(4850, 5)

In [9]:
%%time
textprepare = TextPrepareClass(PHONES, stopwords.words('russian'))
df['text_cl'] = df.review.map(textprepare.clean_all)
df['target']  = df.rating.apply(lambda x: 0 if int(x) >= 4 else 1)

df = df.sample(frac=1).reset_index(drop=True)
df.sample(3)[['review', 'rating', 'link', 'target', 'text_cl']]

CPU times: user 51.1 s, sys: 160 ms, total: 51.2 s
Wall time: 51.2 s


,review,rating,link,target,text_cl
3389,"Не зря Apple считают одной из лучших фирм, пот...",5.0,https://irecommend.ru/content/moya-prelest-282,0,зря appl счита одн лучш фирм техник действител...
1668,Купила это телефон примерно год назад за 5000 ...,1.0,https://irecommend.ru/content/polnyi-uzhas-foto,1,куп эт телефон примерн год назад рубл снача не...
1883,Вся моя семья фанаты сотовых телефонов фирмы s...,5.0,https://irecommend.ru/content/moya-lyubov-k-sm...,0,вся сем фанат сотов телефон фирм samsung одн в...


In [10]:
df['target'].value_counts(normalize=True)

target
0    0.548454
1    0.451546
Name: proportion, dtype: float64

## Создание и сохранение модели и токенайзера для инференса через   
## onnx + docker + flask

Очищенные отзывы векторизую через tf-idf.   
Полученные векторы в LogReg для подбора параметров.

In [11]:
grid_pipe = Pipeline([('vector', TfidfVectorizer(analyzer = 'char_wb')),
                     ('model', LogisticRegression(solver = 'liblinear',))
                     ])

params_pipe = {'vector__max_features': [768, 1024],       # 256, 512, 768, 
               'vector__ngram_range': [(2, 3), (3, 4) ],  # (2, 4)
               'vector__min_df': [0.2, 0.3],              # 0.5
               'vector__max_df': [0.75, 0.8],             # 0.7, 1.0
               'model__penalty': ['l2'],                  # 'l1'
               'model__class_weight': ['balanced'],   #{0: 0.5, 1: 0.5}, {0: 0.5894081632653061, 1: 0.41059183673469385},
               'model__max_iter': [ 20, 100],             # 10, 50],
               'model__C': [2, 3, 4]
               }

In [12]:
%%time
clf = GridSearchCV(grid_pipe, params_pipe,
                   n_jobs=-1, cv=5,
                   scoring = ['roc_auc', 'accuracy'],
                   refit='roc_auc',
                   verbose=0,
                  )
clf.fit(df.text_cl, df.target)
clf.best_score_, clf.best_params_

CPU times: user 7.7 s, sys: 1.49 s, total: 9.19 s
Wall time: 9min 4s


(np.float64(0.9167361897895423),
 {'model__C': 4,
  'model__class_weight': 'balanced',
  'model__max_iter': 20,
  'model__penalty': 'l2',
  'vector__max_df': 0.75,
  'vector__max_features': 1024,
  'vector__min_df': 0.2,
  'vector__ngram_range': (2, 3)})

 Сохраняю результаты на всех возможных комбинациях заданных параметров   
 для дальнейшего анализа

In [21]:
res = pd.DataFrame(clf.cv_results_['params'])
res['mean_test_roc_auc'] = clf.cv_results_['mean_test_roc_auc']
res['mean_test_accuracy'] = clf.cv_results_['mean_test_accuracy']
res['mean_sum'] = (clf.cv_results_['mean_test_accuracy'] +\
                   clf.cv_results_['mean_test_roc_auc']
                   )/2

res.to_csv(os.path.join(PATH_DATA, 'reslt_grid1_logreg.csv'), index=False)

## Обучаю модель на лучших параметрах и полных данных

In [14]:
%%time
vectorizer = TfidfVectorizer(analyzer = 'char_wb',
                             ngram_range=clf.best_params_['vector__ngram_range'],
                             max_df=clf.best_params_['vector__max_df'],
                             min_df=clf.best_params_['vector__min_df'],
                             max_features=clf.best_params_['vector__max_features'],
                            )
vectorizer.fit(df.text_cl)
train = vectorizer.transform(df.text_cl)

model = LogisticRegression(penalty=clf.best_params_['model__penalty'],
                           solver='liblinear',
                           C=clf.best_params_['model__C'],
                           class_weight=clf.best_params_['model__class_weight'],
                           max_iter=clf.best_params_['model__max_iter'],
                          )
model.fit(train, df.target)

CPU times: user 7.78 s, sys: 24.2 ms, total: 7.8 s
Wall time: 7.8 s


LogisticRegression(C=4, class_weight='balanced', max_iter=20,
                   solver='liblinear')

## Сохраняю

С учетом предполагаемого применения, будет необходимо отслеживать версии    
скриптов очистки, приведения к начальной форме и модели. Для облегчения   
этолй задачи объеденим все в класс. При изменении версий данный класс не   
будет занимать много места, об этом не беспокоюсь.    

In [15]:
with open(os.path.join(PATH_MODL, 'logreg_model.pkl'), 'wb') as fd:
    pkl.dump(model, fd)
    
with open(os.path.join(PATH_MODL, 'logreg_vektorizer.pkl'), 'wb') as fd:
    pkl.dump(vectorizer, fd)

## Посмотрим на результат обучения на полном датасете

In [16]:
pred_train_logreg = model.predict(train)
print( f'roc_auc_score: {roc_auc_score(df.target, pred_train_logreg)}\n'\
       f'accuracy_score: {accuracy_score(df.target, pred_train_logreg)}')

roc_auc_score: 0.8851735503141416
accuracy_score: 0.8863917525773196


In [17]:
cm = confusion_matrix(df.target, pred_train_logreg)
cm

array([[2388,  272],
       [ 279, 1911]])

## Ошибки в предсказаниях тональности обзоров

In [18]:
df['pred_train_logreg'] = pred_train_logreg
error_reviews = df[df['target'] != df['pred_train_logreg']]

In [20]:
tmp = error_reviews.sample()
print(f'True class {tmp.target.item()},\nPredicted class {tmp.pred_train_logreg.item()}\n')
print(tmp.review.item())

True class 1,
Predicted class 0

Год назад из России тетя прислала мне смартфон LA'Fleur GT-5380....
ВАУ!!!
on samyi
Круто точно такой же как по ТВ показывали вау я крутая' подумала я))))) Сначала он мне нравился, он казался очень красивым и сенсорные клавиатуры мне нравились. Но.... Потом я стала понимать что с ним скучно, да это круто ютуб,интернет но почему play marketa нету????????!!!!!????Только samsungapps..фу блина это разрдрожала,я не могла смотреть видео в HD качестве. Не могла в браузере запустить видео, через год только получилось.
Не могла скачать нужные приложения whatsapp или skype, ну и приложения для изучения языков для этого надо было купить их за 0.99$ че за фигня на андроиде бесплатно а на баде нет значит!
?В соц сетях могла только сидеть в одноклассниках, в фейсе,в як,в яху,в чатоне и вк! А самое обидное было то что задавали тупой вопрос который так популярен "А это китайский аппарат? " какой на фиг китайский они че телевизор не смотрят! Сколько показывали.
Зарядки 